# 01 — Data Ingestion Pipeline
## Intelligent Customer Analytics Platform
### Olist Brazilian E-Commerce Dataset
**Purpose:** Load all 9 raw CSV files from Unity Catalog Volume into Spark DataFrames and save as Bronze Delta tables using Medallion Architecture.

## Objectives

This notebook performs the initial data ingestion stage of the Medallion Architecture.

The workflow includes:

• Reading all raw Olist CSV datasets

• Loading data into Spark DataFrames

• Performing initial dataset profiling

• Preparing the foundation for Bronze Delta table creation

In [0]:
# Install required libraries
%pip install delta-spark

In [0]:
# Setup SparkSession and confirm environment
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("Spark Version:", spark.version)
print("Environment ready!")

In [0]:
# Define paths to all 9 Olist CSV files in Unity Catalog Volume
base_path = "/Volumes/workspace/default/olist_raw_data"

customers_path    = f"{base_path}/olist_customers_dataset.csv"
orders_path       = f"{base_path}/olist_orders_dataset.csv"
order_items_path  = f"{base_path}/olist_order_items_dataset.csv"
payments_path     = f"{base_path}/olist_order_payments_dataset.csv"
reviews_path      = f"{base_path}/olist_order_reviews_dataset.csv"
products_path     = f"{base_path}/olist_products_dataset.csv"
sellers_path      = f"{base_path}/olist_sellers_dataset.csv"
geolocation_path  = f"{base_path}/olist_geolocation_dataset.csv"
translation_path  = f"{base_path}/product_category_name_translation.csv"

print("All file paths defined!")
print(f"Base path: {base_path}")

In [0]:
# Load all 9 CSV files into Spark DataFrames
from pyspark.sql import functions as F

# Load each table
customers_df   = spark.read.csv(customers_path,   header=True, inferSchema=True)
orders_df      = spark.read.csv(orders_path,      header=True, inferSchema=True)
order_items_df = spark.read.csv(order_items_path, header=True, inferSchema=True)
payments_df    = spark.read.csv(payments_path,    header=True, inferSchema=True)
reviews_df     = spark.read.csv(reviews_path,     header=True, inferSchema=True)
products_df    = spark.read.csv(products_path,    header=True, inferSchema=True)
sellers_df     = spark.read.csv(sellers_path,     header=True, inferSchema=True)
geolocation_df = spark.read.csv(geolocation_path, header=True, inferSchema=True)
translation_df = spark.read.csv(translation_path, header=True, inferSchema=True)

print("All 9 tables loaded successfully!")

In [0]:
# Profile all 9 tables — row counts and column counts
tables = {
    "customers":   customers_df,
    "orders":      orders_df,
    "order_items": order_items_df,
    "payments":    payments_df,
    "reviews":     reviews_df,
    "products":    products_df,
    "sellers":     sellers_df,
    "geolocation": geolocation_df,
    "translation": translation_df
}

print("Table Profiling Summary")
print("-" * 45)
print(f"{'Table':<20} {'Rows':>10} {'Columns':>10}")
print("-" * 45)

for name, df in tables.items():
    print(f"{name:<20} {df.count():>10} {len(df.columns):>10}")

print("-" * 45)
print("All 9 tables profiled successfully!")

In [0]:
# Add metadata columns to all DataFrames
from pyspark.sql import functions as F
from datetime import date

ingestion_date = str(date.today())

def add_metadata(df, table_name):
    return df.withColumn("ingestion_date", 
                         F.lit(ingestion_date)) \
             .withColumn("source_file", 
                         F.lit(f"olist_{table_name}"))

customers_df   = add_metadata(customers_df,   "customers")
orders_df      = add_metadata(orders_df,       "orders")
order_items_df = add_metadata(order_items_df,  "order_items")
payments_df    = add_metadata(payments_df,     "payments")
reviews_df     = add_metadata(reviews_df,      "reviews")
products_df    = add_metadata(products_df,     "products")
sellers_df     = add_metadata(sellers_df,      "sellers")
geolocation_df = add_metadata(geolocation_df,  "geolocation")
translation_df = add_metadata(translation_df,  "translation")

print("Metadata columns added to all 9 tables!")
print(f"ingestion_date: {ingestion_date}")

In [0]:
# Define Bronze Delta table save path
bronze_path = "/Volumes/workspace/default/olist_raw_data/bronze"

# Save all 9 tables as Bronze Delta tables
print("Saving Bronze Delta tables...")
print("-" * 40)

customers_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/customers")
print("customers saved")

orders_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/orders")
print("orders saved")

order_items_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/order_items")
print("order_items saved")

payments_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/payments")
print("payments saved")

reviews_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/reviews")
print("reviews saved")

products_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/products")
print("products saved")

sellers_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/sellers")
print("sellers saved")

geolocation_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/geolocation")
print("geolocation saved")

translation_df.write.format("delta") \
    .mode("overwrite").save(f"{bronze_path}/translation")
print("translation saved")

print("-" * 40)
print("All 9 Bronze Delta tables saved successfully!")

In [0]:
# Verify Bronze Delta tables with DESCRIBE HISTORY
# This proves Delta Lake time travel is working

from delta.tables import DeltaTable

bronze_path = "/Volumes/workspace/default/olist_raw_data/bronze"

print("Bronze Layer Verification")
print("-" * 50)

# Verify row counts from Delta tables
tables_verification = {
    "customers"  : f"{bronze_path}/customers",
    "orders"     : f"{bronze_path}/orders",
    "order_items": f"{bronze_path}/order_items",
    "payments"   : f"{bronze_path}/payments",
    "reviews"    : f"{bronze_path}/reviews",
    "products"   : f"{bronze_path}/products",
    "sellers"    : f"{bronze_path}/sellers",
    "geolocation": f"{bronze_path}/geolocation",
    "translation": f"{bronze_path}/translation"
}

print(f"{'Table':<15} {'Row Count':>10} {'Status':>10}")
print("-" * 50)

for table_name, path in tables_verification.items():
    df = spark.read.format("delta").load(path)
    count = df.count()
    print(f"{table_name:<15} {count:>10} {'VERIFIED':>10}")

print("-" * 50)
print("All Bronze Delta tables verified successfully!")

In [0]:
# check delta history for orders table to confirm versioning is working

delta_table = DeltaTable.forPath(spark, f"{bronze_path}/orders")
delta_table.history().select(
    "version",
    "timestamp",
    "operation"
).show(truncate=False)